In [ ]:
import re
import regex
assert "regex" in str(regex.compile("(test){e<=1}"))
import pandas as pd
import numpy as np
import torch
import transformers
from accelerate import Accelerator
pd.set_option('display.max_colwidth', 80)
from nltk.corpus import stopwords
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_predict
    )
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    precision_recall_fscore_support, roc_auc_score, f1_score
    )
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
    )
from sentence_transformers import SentenceTransformer
from sklearn.pipeline import Pipeline
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
from pathlib import Path
project_dir = Path().resolve()
os.chdir(project_dir)

In [2]:
df = pd.read_csv('hand_coded_sample_post_cor.csv')
print(len(df))
df = df.drop(df[df['shelly_notes'] == "Appeal"].index)
print(len(df))
for col in ["complaint_clean", "findings_clean"]:
    df[col] = (
        df[col]
        .str.replace(r"\bcpoa\b", "agency", regex=True, case=False)
        .str.replace(r"\bapd\b", "police", regex=True, case=False)
    )
df['snippet_clean'] = (
    df["complaint_clean"].fillna("").str.strip()
    + "\n"
    + df["findings_clean"].fillna("").str.strip()
).str.strip()

190
190


In [5]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256)

# Evaluation function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = logits[:, 1]
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary")
    macro_f1 = f1_score(labels, preds, average='macro')
    auc = roc_auc_score(labels, probs)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "macro_f1": macro_f1,
        "auc": auc}

# Setting training arguments
training_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)


# Making Weighted Trainer class
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

## MH SNIPPET

In [6]:
TARGET = "mental_health"
TEXT_COLUMN = "snippet_clean"

In [7]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [8]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.704400,0.586254,0.947368,0.000000,0.000000,0.000000,0.486486,0.722222
2,0.560900,0.588072,0.947368,0.000000,0.000000,0.000000,0.486486,0.736111
3,0.616400,0.588818,0.947368,0.000000,0.000000,0.000000,0.486486,0.750000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then

TrainOutput(global_step=57, training_loss=0.6337865193684896, metrics={'train_runtime': 1533.6222, 'train_samples_per_second': 0.297, 'train_steps_per_second': 0.037, 'total_flos': 59989320622080.0, 'train_loss': 0.6337865193684896, 'epoch': 3.0})

In [10]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.5888180732727051,
 'eval_accuracy': 0.9473684210526315,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4864864864864865,
 'eval_auc': 0.75,
 'eval_runtime': 35.6462,
 'eval_samples_per_second': 1.066,
 'eval_steps_per_second': 0.14,
 'epoch': 3.0}

In [11]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## MH COMPLAINT

In [12]:
TARGET = "mental_health"
TEXT_COLUMN = "complaint_clean"

In [13]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [14]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.501700,0.592407,0.947368,0.000000,0.000000,0.000000,0.486486,0.777778
2,0.537400,0.632383,0.947368,0.000000,0.000000,0.000000,0.486486,0.736111
3,0.714100,0.690765,0.947368,0.000000,0.000000,0.000000,0.486486,0.666667


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then

TrainOutput(global_step=57, training_loss=0.6095724356801886, metrics={'train_runtime': 1378.5467, 'train_samples_per_second': 0.331, 'train_steps_per_second': 0.041, 'total_flos': 59989320622080.0, 'train_loss': 0.6095724356801886, 'epoch': 3.0})

In [15]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.690765380859375,
 'eval_accuracy': 0.9473684210526315,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4864864864864865,
 'eval_auc': 0.6666666666666667,
 'eval_runtime': 29.8028,
 'eval_samples_per_second': 1.275,
 'eval_steps_per_second': 0.168,
 'epoch': 3.0}

In [16]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## BIAS SNIPPET

In [17]:
TARGET = "bias_complaint"
TEXT_COLUMN = "snippet_clean"

In [18]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [19]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.597700,0.649940,0.947368,0.000000,0.000000,0.000000,0.486486,0.402778
2,0.575600,0.796711,0.947368,0.000000,0.000000,0.000000,0.486486,0.388889
3,0.900800,0.876458,0.947368,0.000000,0.000000,0.000000,0.486486,0.513889


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then

TrainOutput(global_step=57, training_loss=0.6796399794126812, metrics={'train_runtime': 1368.4365, 'train_samples_per_second': 0.333, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6796399794126812, 'epoch': 3.0})

In [20]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.8764581084251404,
 'eval_accuracy': 0.9473684210526315,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4864864864864865,
 'eval_auc': 0.5138888888888888,
 'eval_runtime': 29.9757,
 'eval_samples_per_second': 1.268,
 'eval_steps_per_second': 0.167,
 'epoch': 3.0}

In [21]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## BIAS COMPLAINT

In [22]:
TARGET = "bias_complaint"
TEXT_COLUMN = "complaint_clean"

In [23]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [24]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.631800,0.655601,0.947368,0.000000,0.000000,0.000000,0.486486,0.819444
2,0.608200,0.665956,0.947368,0.000000,0.000000,0.000000,0.486486,0.638889
3,0.891500,0.789408,0.947368,0.000000,0.000000,0.000000,0.486486,0.736111


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then

TrainOutput(global_step=57, training_loss=0.6898875529305977, metrics={'train_runtime': 1365.7431, 'train_samples_per_second': 0.334, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6898875529305977, 'epoch': 3.0})

In [25]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.7894075512886047,
 'eval_accuracy': 0.9473684210526315,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4864864864864865,
 'eval_auc': 0.736111111111111,
 'eval_runtime': 29.96,
 'eval_samples_per_second': 1.268,
 'eval_steps_per_second': 0.167,
 'epoch': 3.0}

In [26]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## PROPERTY SNIPPET

In [27]:
TARGET = "property_complaint"
TEXT_COLUMN = "snippet_clean"

In [28]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [29]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.726900,0.680754,0.921053,0.000000,0.000000,0.000000,0.479452,0.542857
2,0.564500,0.686041,0.921053,0.000000,0.000000,0.000000,0.479452,0.533333
3,0.632200,0.689240,0.921053,0.000000,0.000000,0.000000,0.479452,0.552381


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then

TrainOutput(global_step=57, training_loss=0.6414991010699356, metrics={'train_runtime': 1358.0019, 'train_samples_per_second': 0.336, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6414991010699356, 'epoch': 3.0})

In [30]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.6892399191856384,
 'eval_accuracy': 0.9210526315789473,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4794520547945205,
 'eval_auc': 0.5523809523809523,
 'eval_runtime': 29.6886,
 'eval_samples_per_second': 1.28,
 'eval_steps_per_second': 0.168,
 'epoch': 3.0}

In [31]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## PROPERTY COMPLAINT

In [32]:
TARGET = "property_complaint"
TEXT_COLUMN = "complaint_clean"

In [33]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [34]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.683000,0.689781,0.921053,0.000000,0.000000,0.000000,0.479452,0.542857
2,0.547300,0.699279,0.921053,0.000000,0.000000,0.000000,0.479452,0.571429
3,0.641700,0.700552,0.921053,0.000000,0.000000,0.000000,0.479452,0.571429


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then

TrainOutput(global_step=57, training_loss=0.6318395430581611, metrics={'train_runtime': 1355.3373, 'train_samples_per_second': 0.336, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6318395430581611, 'epoch': 3.0})

In [35]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.7005524635314941,
 'eval_accuracy': 0.9210526315789473,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4794520547945205,
 'eval_auc': 0.5714285714285714,
 'eval_runtime': 29.6246,
 'eval_samples_per_second': 1.283,
 'eval_steps_per_second': 0.169,
 'epoch': 3.0}

In [36]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## DEMEANOR SNIPPET

In [37]:
TARGET = "demeanor_complaint"
TEXT_COLUMN = "snippet_clean"

In [38]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [39]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.752300,0.692805,0.447368,0.370370,0.714286,0.487805,0.443902,0.589286
2,0.667900,0.707603,0.368421,0.352941,0.857143,0.500000,0.321429,0.556548
3,0.661200,0.689775,0.421053,0.318182,0.500000,0.388889,0.419444,0.613095


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=57, training_loss=0.6859605772453442, metrics={'train_runtime': 1353.8897, 'train_samples_per_second': 0.337, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6859605772453442, 'epoch': 3.0})

In [40]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6897754669189453,
 'eval_accuracy': 0.42105263157894735,
 'eval_precision': 0.3181818181818182,
 'eval_recall': 0.5,
 'eval_f1': 0.3888888888888889,
 'eval_macro_f1': 0.41944444444444445,
 'eval_auc': 0.6130952380952381,
 'eval_runtime': 29.6194,
 'eval_samples_per_second': 1.283,
 'eval_steps_per_second': 0.169,
 'epoch': 3.0}

In [41]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.56      0.38      0.45        24
           1       0.32      0.50      0.39        14

    accuracy                           0.42        38
   macro avg       0.44      0.44      0.42        38
weighted avg       0.47      0.42      0.43        38



## DEMEANOR COMPLAINT

In [42]:
TARGET = "demeanor_complaint"
TEXT_COLUMN = "complaint_clean"

In [43]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [44]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.732900,0.704952,0.342105,0.322581,0.714286,0.444444,0.318996,0.529762
2,0.679100,0.692971,0.447368,0.379310,0.785714,0.511628,0.437632,0.541667
3,0.626900,0.679530,0.684211,0.750000,0.214286,0.333333,0.563218,0.559524


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=57, training_loss=0.6753937068738436, metrics={'train_runtime': 1347.3706, 'train_samples_per_second': 0.338, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6753937068738436, 'epoch': 3.0})

In [45]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6795298457145691,
 'eval_accuracy': 0.6842105263157895,
 'eval_precision': 0.75,
 'eval_recall': 0.21428571428571427,
 'eval_f1': 0.3333333333333333,
 'eval_macro_f1': 0.5632183908045977,
 'eval_auc': 0.5595238095238094,
 'eval_runtime': 29.8225,
 'eval_samples_per_second': 1.274,
 'eval_steps_per_second': 0.168,
 'epoch': 3.0}

In [46]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.68      0.96      0.79        24
           1       0.75      0.21      0.33        14

    accuracy                           0.68        38
   macro avg       0.71      0.59      0.56        38
weighted avg       0.70      0.68      0.62        38



## REGULATORY SNIPPET

In [47]:
TARGET = "regulatory_complaint"
TEXT_COLUMN = "snippet_clean"

In [48]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [49]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.839400,0.708237,0.631579,0.111111,0.142857,0.125000,0.445833,0.364055
2,0.692300,0.715811,0.578947,0.090909,0.142857,0.111111,0.417625,0.410138
3,0.622900,0.726347,0.552632,0.083333,0.142857,0.105263,0.403509,0.451613


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=57, training_loss=0.705975005501195, metrics={'train_runtime': 1359.5131, 'train_samples_per_second': 0.335, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.705975005501195, 'epoch': 3.0})

In [50]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.7263468503952026,
 'eval_accuracy': 0.5526315789473685,
 'eval_precision': 0.08333333333333333,
 'eval_recall': 0.14285714285714285,
 'eval_f1': 0.10526315789473684,
 'eval_macro_f1': 0.40350877192982454,
 'eval_auc': 0.45161290322580644,
 'eval_runtime': 30.0051,
 'eval_samples_per_second': 1.266,
 'eval_steps_per_second': 0.167,
 'epoch': 3.0}

In [51]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.77      0.65      0.70        31
           1       0.08      0.14      0.11         7

    accuracy                           0.55        38
   macro avg       0.43      0.39      0.40        38
weighted avg       0.64      0.55      0.59        38



## REGULATORY COMPLAINT

In [52]:
TARGET = "regulatory_complaint"
TEXT_COLUMN = "complaint_clean"

In [53]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [54]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.836800,0.691843,0.815789,0.000000,0.000000,0.000000,0.449275,0.382488
2,0.651600,0.686271,0.815789,0.000000,0.000000,0.000000,0.449275,0.327189
3,0.726600,0.680943,0.815789,0.000000,0.000000,0.000000,0.449275,0.387097


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then

TrainOutput(global_step=57, training_loss=0.7052271658914131, metrics={'train_runtime': 1359.9802, 'train_samples_per_second': 0.335, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.7052271658914131, 'epoch': 3.0})

In [55]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.6809426546096802,
 'eval_accuracy': 0.8157894736842105,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4492753623188406,
 'eval_auc': 0.3870967741935484,
 'eval_runtime': 29.6598,
 'eval_samples_per_second': 1.281,
 'eval_steps_per_second': 0.169,
 'epoch': 3.0}

In [56]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## TRAFFIC SNIPPET

In [57]:
TARGET = "traffic_scenario"
TEXT_COLUMN = "snippet_clean"

In [58]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [59]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.769600,0.693106,0.236842,0.194444,1.000000,0.325581,0.223397,0.474654
2,0.689400,0.691057,0.815789,0.000000,0.000000,0.000000,0.449275,0.663594
3,0.701700,0.687678,0.789474,0.000000,0.000000,0.000000,0.441176,0.548387


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=57, training_loss=0.6999886161402652, metrics={'train_runtime': 1346.6283, 'train_samples_per_second': 0.339, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6999886161402652, 'epoch': 3.0})

In [60]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6876780986785889,
 'eval_accuracy': 0.7894736842105263,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_macro_f1': 0.4411764705882353,
 'eval_auc': 0.5483870967741935,
 'eval_runtime': 29.6943,
 'eval_samples_per_second': 1.28,
 'eval_steps_per_second': 0.168,
 'epoch': 3.0}

In [61]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.81      0.97      0.88        31
           1       0.00      0.00      0.00         7

    accuracy                           0.79        38
   macro avg       0.41      0.48      0.44        38
weighted avg       0.66      0.79      0.72        38



## TRAFFIC COMPLAINT

In [62]:
TARGET = "traffic_scenario"
TEXT_COLUMN = "complaint_clean"

In [63]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2)

# train-test split stratified by variable to account for class imbalance
train_df, test_df = train_test_split(
    df.dropna(subset=[TARGET]),
    test_size=0.2,
    random_state=123,
    stratify=df.dropna(subset=[TARGET])[TARGET])

# formatting, tokenizing, formatting more
train_dataset = Dataset.from_pandas(
    train_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
test_dataset = Dataset.from_pandas(
    test_df[[TEXT_COLUMN, TARGET]].rename(
        columns={TEXT_COLUMN: "text", TARGET: "label"}
    ))
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])
train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df[TARGET])

class_weights = torch.tensor(class_weights, dtype=torch.float)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

In [64]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Macro F1,Auc
1,0.751100,0.695305,0.526316,0.210526,0.571429,0.307692,0.473846,0.677419
2,0.672700,0.701557,0.815789,0.000000,0.000000,0.000000,0.449275,0.474654
3,0.711300,0.701370,0.605263,0.100000,0.142857,0.117647,0.431705,0.571429


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=57, training_loss=0.6927245625278406, metrics={'train_runtime': 1354.2763, 'train_samples_per_second': 0.337, 'train_steps_per_second': 0.042, 'total_flos': 59989320622080.0, 'train_loss': 0.6927245625278406, 'epoch': 3.0})

In [65]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.70136958360672,
 'eval_accuracy': 0.6052631578947368,
 'eval_precision': 0.1,
 'eval_recall': 0.14285714285714285,
 'eval_f1': 0.11764705882352941,
 'eval_macro_f1': 0.4317048853439681,
 'eval_auc': 0.5714285714285714,
 'eval_runtime': 29.5913,
 'eval_samples_per_second': 1.284,
 'eval_steps_per_second': 0.169,
 'epoch': 3.0}

In [66]:
pred_output = trainer.predict(test_dataset)
pred_labels = np.argmax(pred_output.predictions, axis=1)

print(
    classification_report(
        test_df[TARGET],
        pred_labels
    )
)

df.loc[test_df.index, f"{TARGET}_BERTpred_{TEXT_COLUMN}"] = pred_labels


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.79      0.71      0.75        31
           1       0.10      0.14      0.12         7

    accuracy                           0.61        38
   macro avg       0.44      0.43      0.43        38
weighted avg       0.66      0.61      0.63        38

